# Transformers Neural Network

## Self-Attention
Self-Attention is a sequence to sequence operation. Consider $x_1, x_2, \cdots, x_t$ input vectors and the corresponding $t$ output vectors $y_1, y_2, \cdots, y_{t}$ (all vectors have dimensions $k$). The output vectors are produced as follows:

$$
y_i = \sum_{j=1}^{t} w_{ij}x_j
$$

Note that $w_{ij}$ is not a learnable parameter, but it's also computed using input vectors as follows:

$$
w_{ij} = \text{SoftMax}(x_i x_j^T)
$$

If we look carefully, input vector $x_i$ is used in three different ways:
- **Query**: Its compared to every other vector to establish weights for its own output $y_i$.
- **Keys**: Its compared to every other vector to establish weights for the output $y_j$.
- **Values**: Its used as part of the weighted sum to compute each output vector once all the weights have been established.

How about we derive new vectors for these three roles using a Neural Network and then perform self-attention?

$$
\mathbf{q_i} = \text{NN}_q(x_i) \ \ \ \ \ \mathbf{k_i} = \text{NN}_k(x_i) \ \ \ \ \ \mathbf{v_i} = \text{NN}_v(x_i)
$$

$$
w_{ij} = \text{SoftMax}(\mathbf{q_i} \mathbf{k_j}^T)
$$

$$
y_i = \sum_{j=1}^{t} w_{ij}\mathbf{v_j}
$$

What this does is adds learnable parameters which can modify the incoming vectors to suit the three roles!

The softmax function can be sensitive to very large input values. These kill the gradient, and slow down learning, or cause it to stop altogether. Since the average value of the dot product grows with the embedding dimension $k$, it helps to scale the dot product back a little to stop the inputs to the softmax function from growing too large:

$$
w_{ij} = \text{SoftMax}(\frac{\mathbf{q_i} \mathbf{k_j}^T}{\sqrt{k}})
$$

## Multi-head Attention
To add flexibility, why not use multiple self attentions in parallel?

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

import math

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, k, heads):
        super().__init__()
        
        # Embedding Dims must be divisible by the number of heads
        assert k % heads == 0
        
        self.k, self.heads = k, heads

        # Computing queries, keys and values for all heads
        self.to_keys = nn.Linear(k, k)
        self.to_queries = nn.Linear(k, k)
        self.to_values = nn.Linear(k, k)

        # To collapse the dims after multihead operations
        self.unify_heads = nn.Linear(k, k)

    def forward(self, x):
        # Batches, num_input_vecs, dim_vec
        b, t, k = x.shape
        
        # Retrieve the number of heads
        h = self.heads

        # Compute queries, keys and values
        queries = self.to_queries(x)
        keys = self.to_keys(x)
        values = self.to_values(x)

        # Dividing the full sequences into h chunks for efficient multihead attention
        s = k / h

        queries = queries.reshape(b, t, h, s)
        keys = keys.reshape(b, t, h, s)
        values = values.reshape(b, t, h, s)

        # Collapse batches and heads into a single dimension
        queries = queries.transpose(-2, -1).reshape(b*h, t, s)
        keys = keys.transpose(-2, -1).reshape(b*h, t, s)
        values = values.transpose(-2, -1).reshape(b*h, t, s)

        # Computing Attention
        w = F.softmax(torch.bmm(queries, keys.transpose(-2, -1)) / (math.sqrt(k)), dim=-1)
        y = torch.bmm(w, values).reshape(b, h, t, s)

        # Unify Heads
        y = y.transpose(-2, -1).reshape(b, t, s*h)
        y = self.unify_heads(y)

        return y

## Transformer Block

In [3]:
class TransformerBLock(nn.Module):
    def __init__(self, k, heads):
        super().__init__()

        # Multihead Self Attention
        self.multihead_attention = MultiHeadAttention(k, heads)

        # Normalisation Layers
        self.norm1 = nn.LayerNorm(k)
        self.norm2 = nn.LayerNorm(k)

        # FFNs
        self.ffn = nn.Sequential(
            nn.Linear(k, 4*k),
            nn.ReLU(),
            nn.Linear(4*k, k)
        )

    def forward(self, x):
        # Multiheaded Attention Applied
        y = self.multihead_attention(x)

        # Normalisation
        x = self.norm1(y + x)

        # FFNs
        a = self.ffn(x)

        # Normalisation
        out = self.norm2(a + x)

        return out

# Classification Transformer

## Dataset

In [11]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw_datasets = load_dataset("imdb")

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

(…)cased/resolve/main/tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

(…)bert-base-cased/resolve/main/config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

(…)o/bert-base-cased/resolve/main/vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

(…)t-base-cased/resolve/main/tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
raw_datasets["train"]["text"][0]

'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, ev

In [20]:
tokenized_datasets["train"]["input_ids"][0]

tensor([  101,   146, 12765,   146,  6586,   140, 19556, 19368, 13329,   118,
          162, 21678,  2162, 17056,  1121,  1139,  1888,  2984,  1272,  1104,
         1155,  1103,  6392,  1115,  4405,  1122,  1165,  1122,  1108,  1148,
         1308,  1107,  2573,   119,   146,  1145,  1767,  1115,  1120,  1148,
         1122,  1108,  7842,  1118,   158,   119,   156,   119, 10148,  1191,
         1122,  1518,  1793,  1106,  3873,  1142,  1583,   117,  3335,  1217,
          170,  5442,  1104,  2441,  1737,   107,  6241,   107,   146,  1541,
         1125,  1106,  1267,  1142,  1111,  1991,   119,   133,  9304,   120,
          135,   133,  9304,   120,   135,  1109,  4928,  1110,  8663,  1213,
          170,  1685,  3619,  3362,  2377,  1417, 14960,  1150,  3349,  1106,
         3858,  1917,  1131,  1169,  1164,  1297,   119,  1130,  2440,  1131,
         3349,  1106,  2817,  1123,  2209,  1116,  1106,  1543,  1199,  3271,
         1104,  4148,  1113,  1184,  1103,  1903,   156, 11547, 

In [28]:
tokenized_datasets["train"]["input_ids"][0].shape, len(raw_datasets["train"]["text"][0])

(torch.Size([512]), 1640)